## Metoda Czasu Urojonego 
- Dla równania Schrödingera niezależnego do czasu w 1D i 2D

In [ ]:
# Blokada wielowątkowości
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
# Biblioteki
import numpy as np
from time import perf_counter
import matplotlib.pyplot as plt
from scipy.sparse import diags # Do rzadkiej macierzy
from scipy.sparse.linalg import eigsh

In [ ]:
# Klasa 1. do pomiaru czasu.
class Timer():
    def __init__(self, name):
        self.name = name
        self.engine_id = os.environ.get('IPY_ENGINE_ID', '0')
    
    def __enter__(self):
        self.start = perf_counter()
        return self

    def __exit__(self, *args):
        self.end = perf_counter()
        if self.engine_id == '0':
            t = self.end - self.start
            print(f"Time - {self.name}: {t/60:.3f} min")

In [ ]:
# Stałe fizyczne.
PI = np.pi
h_bar = 1.0
m = 1.0

In [ ]:
# Funkcja 1. do energii analitycznie
def energy_analytical(N, L, D):
    x = np.linspace(0, L, N)
    if D == 1:
        psi = np.sqrt(2/L) * np.sin(PI * x / L)
        e = (PI**2 * h_bar**2) / (2.0 * m * L**2)
        return psi, e, x
    if D == 2:
        y = np.linspace(0, L, N)
        X, Y = np.meshgrid(x, y)
        psi = (2/L) * np.sin(PI * X / L) * np.sin(PI * Y / L)
        e = (PI**2 * h_bar**2) / (m * L**2)
        return psi, e, x, y
    print("ERROR energy_analytical")
    return None

In [ ]:
# Funkcja 2. do inicjalizacji Hamiltonianu i stałych układu.
def initialize(N, L, v_func, D):
    dx = L / (N-1)
    x = np.linspace(0, L, N)
#-------------------------------------------------------------------------------------------------------------------
    if(D==1):
        dtau = 0.1 * m*dx**2 / h_bar
        psi = np.zeros(N)
        v = np.zeros(N)
        for i in range(N):
            v[i] = v_func(x[i], None, L, D)
            psi[i] = x[i]*(L - x[i])
        return dx, dtau, psi, v,  x
#-------------------------------------------------------------------------------------------------------------------
    if(D==2):
        dy = dx
        dtau = 0.1* m*dx*dy / (2.0*h_bar)
        y = np.zeros(N)
        for i in range(1, N): y[i] = y[i-1]+dy
        psi = np.zeros((N, N))
        v = np.zeros((N, N))
        for i in range(N):
            for j in range(N):
                v[i][j] = v_func(x[i], y[j], L, D)
                psi[i][j] = x[i]*(L - x[i]) + y[j]*(L - y[j])
        return dx, dtau, psi, v, x, y
    print("ERROR initialize")
    return None

In [ ]:
# Funkcje 3. potencjałów
def V_infinite(x, y, L, D):
    return 0.0
#-------------------------------------------------------------------------------------------------------------------
def V_finite(x, y, L, D): # Studnia w centrum od L/4 do 3L/4
    a = L / 2.0
    V0 = -10.0 # Głębokość studni
    if(D == 1):
        if(L/2.0 - a/2.0) <= x <= (L/2.0 + a/2.0): return V0
        return 0.0
    if (D == 2):
        if (L/2.0 - a/2.0) <= x <= (L/2.0 + a/2.0) and (L/2.0 - a/2.0) <= y <= (L/2.0 + a/2.0): return V0
        return 0.0
#-------------------------------------------------------------------------------------------------------------------
def V_gauss1(x, y, L, D):
    V0 = -100.0
    sigma = L / 8.0
    return V0 * np.exp(-((x - L/2.0)**2 + (y - L/2.0)**2) / sigma**2)

In [ ]:
# Funkcja 4. energii z równ. Schrodingera
def Energy(N, psi, v, dx, D):
    e = 0.0
    if(D == 1):
        for i in range(1, N-1):
            kinetic = -h_bar**2 * (psi[i-1] - 2.0*psi[i] + psi[i+1])\
                        / (2.0 * m * dx**2)
            potential = v[i] * psi[i]
            e += psi[i] * (kinetic + potential) * dx
    if(D == 2):
        for i in range(1, N-1):
                for j in range(1, N-1):
                    kinetic = -h_bar**2 * (psi[i-1][j] + psi[i+1][j] + psi[i][j-1] + psi[i][j+1] - 4.0*psi[i][j])\
                                / (2.0 * m * dx**2)
                    potential =  v[i][j] * psi[i][j] 
                    e += psi[i][j] * (kinetic + potential) * dx**2
    return e

In [ ]:
# Funkcja 5. do normalizacji f. falowej.
def psi_norm(psi, dx, D):
    if (D == 1):
        norm = np.sqrt(np.sum(psi**2) * dx)
    if (D == 2):
        norm = np.sqrt(np.sum(psi**2) * dx**2)
    return psi / norm

In [ ]:
# Funkcja 6. krok metody urojonej
def step_imaginary(psi, v, i, j, dx, dtau, D):
    if (D == 1):
        kinetic = -h_bar**2 * (psi[i-1] - 2.0*psi[i] + psi[i+1])\
                            / (2.0 * m * dx**2)
        potential =  v[i]*psi[i]
        return psi[i] - dtau*(kinetic + potential)
    if (D == 2):
        kinetic = -h_bar**2 * (psi[i-1][j] + psi[i+1][j] + psi[i][j-1] + psi[i][j+1] - 4.0*psi[i][j])\
                                / (2.0 * m * dx**2)
        potential =  v[i][j]*psi[i][j]
        return psi[i][j] - dtau/h_bar * (kinetic + potential)
    print("ERROR step_imaginary")
    return 1

# Komentarz 1.
- Dokładność metody czasu urojonego epsilon = 1e-7

In [ ]:
# Funkcja 7. do metody czasu urojonego.
def imaginary(N, _L, _v_func, _D):
    with Timer(f'Imaginary N={N}, L={_L}'):
        p=0
        epsilon = 1e-7 #-10
        difference = 1.0
        E_new = 1000.0
        
        if (_D == 1):
            dx, dtau, PSI, V, X = initialize(N, _L, _v_func, _D)
            PSI_new = np.zeros(N)
            
        if (_D == 2): 
            dx, dtau, PSI, V, X, Y = initialize(N, _L, _v_func, _D)
            PSI_new = np.zeros((N, N))
        E_old = Energy(N, PSI, V, dx, _D)
        PSI = psi_norm(PSI, dx, _D) # Normalizacja początkowa
        
        if (_D == 1):
            while (difference > epsilon):
                for i in range(1, N-1):
                    PSI_new[i] = step_imaginary(PSI, V, i, None, dx, dtau, _D)  
                PSI_new[0] = PSI_new[N-1] = 0.0
                PSI = psi_norm(PSI_new, dx, _D)
                E_new = Energy(N, PSI, V, dx, _D)
                difference = abs(E_new - E_old)
                E_old = E_new; p+=1
            return PSI, E_new, X, p
        if (_D == 2):
            while (difference > epsilon):
                for i in range(1, N-1):
                    for j in range(1, N-1):
                        PSI_new[i][j] = step_imaginary(PSI, V, i, j, dx, dtau, _D) 
                PSI_new[0,:] = PSI_new[-1,:] = PSI_new[:,0] = PSI_new[:,-1] = 0.0
                PSI = psi_norm(PSI_new, dx, _D)
                E_new = Energy(N, PSI, V, dx, _D)
                difference = abs(E_new - E_old)
                E_old = E_new; p+=1
            return PSI, E_new, X, Y, p
        print("ERROR imaginary")
        return 1

# Zad 2.2.1
- Stan podstawowy w 1D nieskończonej studni.
- Stabilność(dt, dx), gdzie:
  - dx = L / (N-1)
  - 1D: dtau = 0.1 * m*dx**2 / h_bar
  - 2D: dtau = 0.1* m*dx*dy / (2.0*h_bar)
- Czyli tak naprawdę stabilność w zależności od $\frac{L}{N}$, gdzie:
  - L parametr rozmiaru obszaru
  - N liczba punktów w wierszu siatki
- Uwaga analityczne rozwiązanie nie zależy od N, a przynajmniej nie jakościowo, ale i tak będę porównywał dla tych samych N.

In [ ]:
def plot_E_and_Psi(N, L, v_func, D):
    Psinum, Enum, xnum, iter = imaginary(N=N, _L=L, _v_func=v_func, _D=D)
    Psiana, Eana, xana  = energy_analytical(N=N, L=L, D=D)
    plt.figure(figsize=(4, 3))
    dif_E=abs(Enum-Eana)/Eana*100
    print(f'Błąd względny $E$ w maksimum ={dif_E:.2f}%')
    dif_Psi=abs(max(Psinum)-max(Psiana))/max(Psiana)*100
    print(f'Błąd względny Psi w maksimum ={dif_Psi:.2f}%')
    plt.title("Gęstość prawdopodobieństwa - Nieskończona studnia 1D")
    plt.plot(xnum, Psinum**2, 'red', label=f'Numerical | Iter ={iter}')
    plt.plot(xana, Psiana**2, 'blue', label=f'Analytical')
    plt.xlabel("x")
    plt.ylabel("$|\Psi(x)|^2$"); plt.grid(True)
    plt.legend(loc='upper right'); plt.show()

In [ ]:
plot_E_and_Psi(N=100, L=1.0, v_func=V_infinite, D=1)
plot_E_and_Psi(N=200, L=1.0, v_func=V_infinite, D=1)
plot_E_and_Psi(N=300, L=1.0, v_func=V_infinite, D=1)
plot_E_and_Psi(N=100, L=2.0, v_func=V_infinite, D=1)
plot_E_and_Psi(N=100, L=5.0, v_func=V_infinite, D=1)

In [ ]:
N = [50, 75, 100, 150, 200, 300]
for n in N:
    Psi, E, X, iter = imaginary(n, 1.0, V_infinite, 1)
    _, Eana, _  = energy_analytical(n, 1.0, 1)
    dif = abs(E - Eana)/Eana * 100
    print(f"E = {E:.5f} | Błąd = {dif:.4f}% | Iteracji = {iter}")

# Komentarz 2.
- Wyniki prawidłowo zbiegają dla energii stanu podstawowego do wartości analitycznej wraz ze wzrostem gęstości siatki dyskretyzacji **$dx$**.
- Zaobserwowany przy N=300 nieznaczny wzrost błędu wynika z faktu, że przy bardzo małym kroku przestrzennym narzucony warunek stopu **$\epsilon = 10^{-7}$** tak wpływa na wynik.
- Dodatkowo liczba iteracji rośnie kwadratowo względem N, co jest bezpośrednim skutkiem zachowania stabilności schematu jawnego poprzez skalowanie kroku czasowego **$d\tau \approx dx^2$**

# Zad 2.2.2
- Stan podstawowy w 1D skończonej studni.
- Wyniki dla róźnych L.
- Potencjał **$V_0 = -10$**

In [ ]:
plot_E_and_Psi(N=100, L=0.1, v_func=V_finite, D=1)
plot_E_and_Psi(N=100, L=0.5, v_func=V_finite, D=1)
plot_E_and_Psi(N=100, L=1.0, v_func=V_finite, D=1)
plot_E_and_Psi(N=100, L=1.5, v_func=V_finite, D=1)
plot_E_and_Psi(N=100, L=3.0, v_func=V_finite, D=1)
plot_E_and_Psi(N=100, L=5.0, v_func=V_finite, D=1)

In [ ]:
L = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]
for l in L:
    Psi, E, X, iter = imaginary(100, l, V_finite, 1)
    _, Eana, _  = energy_analytical(100, l, 1)
    dif = abs(E - Eana)/Eana * 100
    print(f"E = {E:.5f} | Odchylenie = {dif:.4f}% | Iteracji = {iter}")

# Komentarz 3.
- Zwiększanie rozmiaru obszaru obliczeniowego **L** przy stałej szerokości studni redukuje wpływ nieskończonych barier potencjału na brzegach, co pozwala funkcji falowej na swobodną penetrację obszaru klasycznie zakazanego i skutkuje obniżeniem energii stanu podstawowego i co możemy zaobserować poprzez co raz to większą różnicę między energią cząstki dla studni nieskończonej a energią dla skończonej studni.

In [ ]:
# Obliczenia 2D
Psinum_2D, Enum_2D, xnum_2D, y_num_2D, iter_2D = imaginary(50, 1.0, V_infinite, 2)
Eana_2D = (PI**2 * h_bar**2) / (m * 1.0**2)
print(f"Energia 2D numeryczna: {Enum_2D:.5f}")
print(f"Energia 2D analityczna: {Eana_2D:.5f}")
plt.figure(figsize=(7, 6))
plt.imshow(Psinum_2D**2)
plt.colorbar(label=f'$|\Psi(x,y)|^2$ \nIter = {iter_2D}')
plt.title("Gęstość prawdopodobieństwa 2D - Stan podstawowy")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# Potencjał Gaussa*
- Przy ujemnym **$V_0 = -100$**​, gęstość prawdopodobieństwa jest jeszcze silniej skupiona w centrum układu (L/2,L/2) niż w przypadku nieskończonej studni, ponieważ potencjał Gaussa dodatkowo "przyciąga" cząstkę do środka.

In [ ]:
# Obliczenia 2D
Psinum_2Dg, Enum_2Dg, xnum_2Dg, ynum_2Dg, iterg_2D = imaginary(50, 1.0, V_gauss1, 2)
Eana_2Dg = (PI**2 * h_bar**2) / (m * 1.0**2)
print(f"Energia 2D numeryczna: {Enum_2Dg:.5f}")
print(f"Energia 2D analityczna: {Eana_2Dg:.5f}")
plt.figure(figsize=(7, 6))
plt.imshow(Psinum_2Dg**2)
plt.colorbar(label=f'$|\Psi(x,y)|^2$ \nIter = {iterg_2D}')
plt.title("Gęstość prawdopodobieństwa 2D - Stan podstawowy")
plt.xlabel("x")
plt.ylabel("y")
plt.show()